<a href="https://colab.research.google.com/github/fbeilstein/bioinformatics/blob/master/practice_03_genetic_polymorphism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Genetic Polymorphism & Disease (Variant Calling)**

In this notebook, we move from evolutionary divergence between species to **genetic polymorphism** within a single species. We will focus on humans and explore how Single Nucleotide Polymorphisms (SNPs) and small Insertions/Deletions (Indels) are identified from raw sequencing data.

We will investigate the **BRCA1** gene (BReast CAncer 1), a critical tumor suppressor gene located on chromosome 17. Pathogenic mutations in this gene significantly increase the risk of breast and ovarian cancers because the cell loses its ability to accurately repair double-strand DNA breaks.

Our goal is to take a raw sequencing dataset from a well-studied human individual (NA12878), call all the genetic variants in the BRCA1 region using a Bayesian algorithm, and identify any potentially pathogenic mutations.


### Part 1: Environment Setup

We will use three standard bioinformatics tools for variant calling and manipulation:
* **samtools**: The Swiss Army knife for reading, writing, and indexing BAM (Binary Alignment Map) files.
* **freebayes**: A haplotype-based Bayesian variant caller. It calculates the probability that a variant exists given the observed read data and base quality scores.
* **cyvcf2**: A fast Python library for parsing VCF (Variant Call Format) files.


In [ ]:
import os
import subprocess
from IPython.display import clear_output

# Install samtools, freebayes, and tabix (for indexing)
!apt-get -y -qq install samtools freebayes tabix

# Install cyvcf2 (Python VCF parser) and other libraries
!pip install -q cyvcf2 pysam matplotlib numpy scipy pandas

clear_output()
print("Installations successful.")

### **Part 2: Dataset Download**

We are analyzing **NA12878**, a thoroughly sequenced and validated sample from the 1000 Genomes Project (often called the "Genome in a Bottle").

Instead of downloading the entire 100+ GB whole-genome dataset, we will dynamically extract a 5 MB slice covering only the BRCA1 locus (chr17:43,044,295-43,170,245) directly from the 1000 Genomes FTP servers. This is possible because BAM files are block-compressed and indexed, allowing `samtools` to retrieve specific genomic coordinates over the network.

We will also download the matching segment of the GRCh38 human reference genome (FASTA), which `freebayes` needs to determine the "baseline" DNA sequence.


In [ ]:
# URLs for the 1000 Genomes Project NA12878 exome BAM and GRCh38 reference FASTA
bam_url = "ftp://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/1000_genomes_project/data/CEU/NA12878/exome_alignment/NA12878.alt_bwamem_GRCh38DH.20150826.CEU.exome.bam"
ref_url = "ftp://ftp.1000genomes.ebi.ac.uk/vol1/ftp/technical/reference/GRCh38_reference_genome/GRCh38_full_analysis_set_plus_decoy_hla.fa"

brca1_region = "chr17:43044295-43170245"

print("Downloading BRCA1 BAM slice... (this may take a minute)")
!samtools view -b -h {bam_url} {brca1_region} > NA12878_BRCA1.bam
!samtools index NA12878_BRCA1.bam

print("Downloading BRCA1 reference FASTA slice...")
!samtools faidx {ref_url} {brca1_region} > brca1_ref.fa
!samtools faidx brca1_ref.fa

clear_output()
print("Data downloaded and indexed.")


### **Part 3: Exploring the BAM File**

A **BAM (Binary Alignment Map)** file stores the sequencing reads that have been mathematically aligned to the reference genome.
Let's inspect the file.
* **Header**: Contains metadata about the reference sequences (chromosomes) and the sequencing run.
* **Alignment Records**: Each line is a single read, showing where it mapped, its CIGAR string (matches, insertions, deletions), and the raw sequence with Phred quality scores.


In [ ]:
# View alignment summary statistics
print("--- Alignment Summary ---")
!samtools flagstat NA12878_BRCA1.bam

print("\n--- First 3 Reads in the BAM file ---")
!samtools view NA12878_BRCA1.bam | head -n 3


**Visualizing Read Depth (Hybridization Capture)**

This dataset comes from **Targeted Exome Sequencing**. Instead of sequencing the whole genome, molecular probes were used to capture only the protein-coding exons.
We can visualize this by plotting the **read depth** (number of overlapping reads) across the locus. You will see "spikes" of high coverage at the exons, separated by "valleys" of zero coverage in the introns.


In [ ]:
import matplotlib.pyplot.subplots
import matplotlib.pyplot as plt
import pandas as pd

# Calculate depth at every position in the locus
!samtools depth NA12878_BRCA1.bam > depth.txt

# Load depth data and plot
depth_df = pd.read_csv("depth.txt", sep="\t", header=None, names=["CHROM", "POS", "DEPTH"])

plt.figure(figsize=(14, 4))
plt.plot(depth_df["POS"], depth_df["DEPTH"], color='black', linewidth=0.5)
plt.title("Read Depth across the BRCA1 Locus (Targeted Exome Capture)")
plt.xlabel("Genomic Position (chr17)")
plt.ylabel("Read Depth")
plt.grid(True, alpha=0.3)
plt.show()


### **Part 4: Bayesian Variant Calling Theory**

Humans are diploid, meaning we have two copies of every chromosome (except sex chromosomes). At any single genomic position, there are three possible **Genotypes ($G$)**:
1. **0/0**: Homozygous Reference (both copies match the standard human reference)
2. **0/1**: Heterozygous (one copy matches the reference, one has a mutation)
3. **1/1**: Homozygous Alternate (both copies have the mutation)

To decide the true genotype, the algorithm uses **Bayes' Theorem**:
$$P(G \mid D) = \frac{P(D \mid G) P(G)}{P(D)} \propto P(D \mid G) P(G)$$

* **$D$ (Data)**: The aligned reads covering the position, including their Phred base quality scores.
* **$P(G)$ (Prior)**: The probability of the genotype occurring in the general population, based on Hardy-Weinberg Equilibrium and estimated allele frequencies.
* **$P(D \mid G)$ (Likelihood)**: The probability of seeing these exact reads if genotype $G$ were true. For example, if the true genotype is heterozygous (0/1), we expect roughly 50% of the reads to show the reference base and 50% to show the alternate base, factoring in the sequencing error probability ($10^{-Q/10}$).
* **$P(G \mid D)$ (Posterior)**: Our confidence in the final genotype call, which is output in the VCF file as the `QUAL` score.


### **Part 5: Variant Calling with freebayes**

We will now run `freebayes` to discover variants. It evaluates the Bayesian posterior probability for every position where the reads mismatch the reference genome.


In [ ]:
# Run freebayes on our BAM slice
# -f: reference fasta
# --min-alternate-fraction: require at least 20% of reads to support the alt allele
!freebayes -f brca1_ref.fa --min-alternate-fraction 0.2 NA12878_BRCA1.bam > brca1_variants.vcf

print("Variant calling complete!")
!grep -v "^#" brca1_variants.vcf | wc -l
print("variants found.")

### **Part 6: VCF Format and Parsing with cyvcf2**

The output is a **VCF (Variant Call Format)** file.
Let's parse it using `cyvcf2` and look at the first few variants. We are specifically looking for the genotype (GT), the total read depth (DP), and the number of reads supporting the Reference (RO) and Alternate (AO) alleles.


In [ ]:
from cyvcf2 import VCF

vcf = VCF("brca1_variants.vcf")

print(f"{'CHROM':<7} | {'POS':<9} | {'REF':<5} | {'ALT':<5} | {'QUAL':<7} | {'GT':<3} | {'DP':<4} | {'RO':<4} | {'AO'}")
print("-" * 75)

count = 0
for variant in vcf:
  if count >= 10:
    break

  chrom = variant.CHROM
  pos = variant.POS
  ref = variant.REF
  alt = variant.ALT[0] if variant.ALT else "."
  qual = round(variant.QUAL, 1) if variant.QUAL is not None else 0

  # Genotype array: [allele1, allele2, is_phased]
  # 0 = ref, 1 = alt1, etc.
  gt_arr = variant.genotypes[0]
  gt = f"{gt_arr[0]}/{gt_arr[1]}"

  # Read depths
  dp = variant.format("DP")[0][0] if variant.format("DP") is not None else 0
  ro = variant.format("RO")[0][0] if variant.format("RO") is not None else 0
  ao = variant.format("AO")[0][0] if variant.format("AO") is not None else 0

  print(f"{chrom:<7} | {pos:<9} | {ref:<5} | {alt:<5} | {qual:<7} | {gt:<3} | {dp:<4} | {ro:<4} | {ao}")
  count += 1


### **Part 7: Biological Interpretation (Pathogenic Variants)**

Many SNPs are benign — they cause no change in protein function, or occur in non-coding regions. However, **missense mutations** (changing an amino acid) or **frameshift indels** in BRCA1 can destroy its ability to repair DNA.

We will cross-reference our called variants with **ClinVar**, a public archive of human genetic variants and their clinically validated phenotypes.


In [ ]:
# A small dictionary of some known pathogenic and benign SNPs in BRCA1 (GRCh38 coordinates)
clinvar_db = {
    43057053: {"significance": "Benign", "disease": "Breast/Ovarian Cancer"},
    43091560: {"significance": "Pathogenic", "disease": "Hereditary breast cancer"},
    43094895: {"significance": "Benign", "disease": "Hereditary breast cancer"},
    43118942: {"significance": "Pathogenic", "disease": "Breast/Ovarian Cancer"},
    43124030: {"significance": "Pathogenic", "disease": "Familial cancer of breast"}
}

vcf = VCF("brca1_variants.vcf")

print("Checking variants against ClinVar database...\n")

found = False
for variant in vcf:
    if variant.POS in clinvar_db:
        db_entry = clinvar_db[variant.POS]
        gt_arr = variant.genotypes[0]
        gt = f"{gt_arr[0]}/{gt_arr[1]}"

        print(f"Variant Found at {variant.CHROM}:{variant.POS} ({variant.REF} > {variant.ALT[0]})")
        print(f"  Sample Genotype:      {gt}")
        print(f"  Clinical Significance: {db_entry['significance']}")
        print(f"  Associated Disease:    {db_entry['disease']}\n")
        found = True

if not found:
    print("No matches found in this small ClinVar subset.")


### **Part 8: Bayesian Calculation by Hand**

Let's demystify `freebayes`. We will take a single variant position, extract the base quality scores of the reads supporting the Reference and Alternate alleles, and manually calculate the Bayesian posterior probability.

$$P(D \mid G=0/0) = \prod_{\text{ref}} (1-\epsilon_i) \prod_{\text{alt}} \epsilon_i$$
$$P(D \mid G=0/1) = \prod_{\text{all reads}} 0.5$$
$$P(D \mid G=1/1) = \prod_{\text{ref}} \epsilon_i \prod_{\text{alt}} (1-\epsilon_i)$$

Where $\epsilon_i$ is the error probability of read $i$: $\epsilon_i = 10^{-Q_i/10}$. We use logarithms to prevent numerical underflow.


In [ ]:
import numpy as np

def compute_likelihoods(ref_quals, alt_quals):
  """Compute log likelihoods for each genotype."""

  # Convert Phred scores to error probabilities
  # (assuming Q is provided directly, but often they are mapped in pysam)
  ref_eps = 10 ** (-np.array(ref_quals) / 10.0)
  alt_eps = 10 ** (-np.array(alt_quals) / 10.0)

  # 0/0 (Hom Ref): Ref reads are correct, Alt reads are errors
  ll_00 = np.sum(np.log(1 - ref_eps)) + np.sum(np.log(alt_eps))

  # 0/1 (Het): True allele is randomly sampled (50% chance of ref, 50% alt)
  # Note: A real caller uses (1 - eps)/2 + eps/2 = 0.5 for both
  ll_01 = (len(ref_quals) + len(alt_quals)) * np.log(0.5)

  # 1/1 (Hom Alt): Alt reads are correct, Ref reads are errors
  ll_11 = np.sum(np.log(ref_eps)) + np.sum(np.log(1 - alt_eps))

  return ll_00, ll_01, ll_11

# Example: A position with 12 Reference reads and 10 Alternate reads
# Let's say all reads have a Phred score of Q30 (0.1% error rate)
ref_quals = [30] * 12
alt_quals = [30] * 10

ll_00, ll_01, ll_11 = compute_likelihoods(ref_quals, alt_quals)

# Convert out of log space for readability (shift by max to avoid underflow)
max_ll = max(ll_00, ll_01, ll_11)
p_00 = np.exp(ll_00 - max_ll)
p_01 = np.exp(ll_01 - max_ll)
p_11 = np.exp(ll_11 - max_ll)

# Normalize (assuming flat prior for this simple demonstration)
total = p_00 + p_01 + p_11
print(f"P(G=0/0 | D) = {p_00/total:.10e}")
print(f"P(G=0/1 | D) = {p_01/total:.10e}")
print(f"P(G=1/1 | D) = {p_11/total:.10e}")
print("\nMost likely genotype: Heterozygous (0/1)")


### **Homework: Mystery Sample Classification & Variant Analysis**

Use the tools and concepts demonstrated above to analyze the VCF file.


#### **Task 1: Quality Filter**

Raw VCF files contain many false positives (e.g., sequencing artifacts, misalignment).
Implement the `filter_variants` function to read `brca1_variants.vcf` and return a list of variants that pass two thresholds:
1. `QUAL >= 20` (99% confidence)
2. `DP >= 15` (At least 15x read depth)


In [ ]:
def filter_variants(vcf_path, min_qual=20, min_depth=15):
  """
  Reads a VCF file and returns a list of variant objects
  that pass the quality and depth filters.
  """
  passed_variants = []

  # ---- YOUR CODE HERE ----

  return passed_variants

# test_variants = filter_variants("brca1_variants.vcf")
# print(f"Passed variants: {len(test_variants)}")


#### **Task 2: Allele Frequency Spectrum**

For every variant that passed your filter in Task 1, calculate the **Alternate Allele Frequency (AAF)** in the sample.
$$AAF = \frac{AO}{RO + AO}$$

Plot a histogram of these AAF values using `matplotlib`.
*Note: You should see peaks around 0.5 (heterozygous) and 1.0 (homozygous alt).*


In [ ]:
# ---- YOUR CODE HERE ----

#### **Task 3: Implement the Bayesian Genotyper**

Expand on the hand-calculation from Part 8. Implement `bayesian_genotyper` which takes arrays of quality scores and a population allele frequency ($f$), applies Hardy-Weinberg priors, and returns the MAP (Maximum A Posteriori) genotype.

**Priors (HWE):**
* $P(0/0) = (1-f)^2$
* $P(0/1) = 2f(1-f)$
* $P(1/1) = f^2$

In [ ]:
def bayesian_genotyper(ref_quals, alt_quals, pop_af=0.01):
  """
  Returns the most likely genotype string ('0/0', '0/1', or '1/1')
  given the quality scores and population allele frequency.
  """

  # ---- YOUR CODE HERE ----

  return "0/0"

#### **Task 4: Validate Against freebayes**

We will now test your Bayesian genotyper against the industrial tool.
Loop through the first 20 passed variants from Task 1. For each variant, extract the base qualities using `pysam`, run your `bayesian_genotyper`, and compare it to the genotype called by `freebayes`.

In [ ]:
import pysam

# Hint: You can open the BAM file with pysam.AlignmentFile
# and use the .fetch(chrom, pos-1, pos) method to get reads covering the variant.

# ---- YOUR CODE HERE ----


#### Task 5: Interpretation Questions

Answer the following in a text cell:

1. A variant has 30 reads: 14 ref, 16 alt. All base qualities are Q30. Calculate $P(G=0/1 \mid D)$ vs. $P(G=1/1 \mid D)$ assuming a population allele frequency $f = 0.01$. Which genotype wins?
2. Why does `freebayes` require a reference FASTA file, not just the BAM?
3. Look at the read depth plot from Part 3. Why is coverage highly uneven across the BRCA1 gene? How does this relate to hybridization capture?
4. What happens to variant calling accuracy when read depth drops below 10x? Why?
5. Why is a population allele frequency prior important? What would change if you used a flat (uninformative) prior like we did in Part 8?